[README](README.md) | [Introduction](Introduction.md) | [Datasets](Datasets.md) | Notebook

# How the Internet assigns and uses Autonomous Systems (ASes)

In [1]:
name = "William Elsen"
date = "06/21/2026"

In [13]:
# Setup: shared utilities and tier definitions
import bz2
import urllib.request
import gzip
import io
import json
import zipfile
from contextlib import contextmanager
from pathlib import Path
from IPython.display import Markdown, display


@contextmanager
def open_safe(filename, encoding="utf-8"):
    """Open a file for reading, transparently handling .gz, .bz2, and .zip compression."""
    path = Path(filename)
    suffix = path.suffix.lower()
    if suffix == ".gz":
        with gzip.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".bz2":
        with bz2.open(path, "rt", encoding=encoding) as f:
            yield f
    elif suffix == ".zip":
        with zipfile.ZipFile(path) as zf:
            with zf.open(zf.namelist()[0]) as raw:
                yield io.TextIOWrapper(raw, encoding=encoding)
    else:
        with path.open(encoding=encoding) as f:
            yield f


TIERS = [
    ("edge",           1,     1),
    ("transit small",  2,     10),
    ("transit middle", 11,    1000),
    ("transit large",  1001,  10000),
    ("transit huge",   10001, -1),
]

---

## Task 2: ASN Classified by Customer Cone Size

Fill in the missing code in the cells below to produce **Table 1**.

### Data format

In `data/20260501.ppdc-ases.txt.bz2`, lines starting with `#` are comments.
All other lines start with an ASN followed by the ASNs in its customer cone.
The cone size for that ASN is the number of space-separated tokens on the line minus 1.

```
# This is a comment
# 23's customer cone size is 3 and includes (23, 4, 1)
23 23 4 1
# 1's customer cone size is 1 — it only includes itself
1 1
```

### Table 1 format

|           tier | range        | number of ASNs |   percentage |
| -------------: | ------------ | -------------: | -----------: |
|           edge | 1            |        [total] | [percentage] |
|  transit small | 2..10        |        [total] | [percentage] |
| transit middle | 11..1000     |        [total] | [percentage] |
|  transit large | 1001..10000  |        [total] | [percentage] |
|   transit huge | 10001..[max] |        [total] | [percentage] |

Replace `[max]` with the actual maximum cone size found in the data.
Percentages are formatted to one decimal place.

In [22]:
AS_CONE_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as-relationships/20260501.ppdc-ases.txt.bz2"
AS_CONE_PATH = Path("data/20260501.ppdc-ases.txt.bz2")

def classify(size: int) -> str:
    if(size==1):
        return TIERS[0][0]
    elif(size <= 10):
        return TIERS[1][0]
    elif(size <= 1000):
        return TIERS[2][0]
    elif(size <= 10000):
        return TIERS[3][0]
    elif(size > 10000):
        return TIERS[4][0]    
    return "unknown"
    # Use the TIERS list to determine which tier 
    # the given size falls into, and return the corresponding label.
    # Return "unknown" if the size does not fit into any tier.

print(f"Downloading {AS_CONE_URL} ...")
AS_CONE_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS_CONE_URL, AS_CONE_PATH)
if not AS_CONE_PATH.exists(): 
    print (f"Unable to save file {AS_CONE_PATH}")
    exit() 

# Build a dict mapping each ASN to its customer cone size
asn_to_size = {} 
with open_safe(AS_CONE_PATH) as fin:
    for line in fin:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        asn_list = line.split()
        size = len(asn_list) - 1
        asn_to_size[asn_list[0]] = size
        # count the number of ASN after the first
        # ASN on the line. 
        # 2 3 45 
        # asn_to_size["2"] = len(["3","45"])
print(f"Number of ASNs: {len(asn_to_size)}")

total = len(asn_to_size)
max_size = max(asn_to_size.values()) if asn_to_size else 0
print(max_size)
tier_counts = {label: 0 for label, _, _ in TIERS}
for size in asn_to_size.values():
    tier_counts[classify(size)] += 1

header = "| tier | range | number of ASNs | percentage |"
sep    = "| ---: | ----- | ---: | ---: |"
table_1_rows = [header, sep]

for label, lo, hi in TIERS:
    range_str = f"{lo}..{hi}"
    count = tier_counts[label]
    pct = count/total * 100
    # Compute the range string, count, and percentage 
    # for this tier, and append a row
    table_1_rows.append(f"| {label} | {range_str} | {count} | {pct:.1f}% |")
table_1 = "\n".join(table_1_rows)

Number of ASNs: 79644
56415


In [21]:
display(Markdown(table_1))
Path("tables").mkdir(exist_ok=True)
Path("tables/asn-cone-tiers.md").write_text(table_1, encoding="utf-8")

| tier | range | number of ASNs | percentage |
| ---: | ----- | ---: | ---: |
| edge | 1..1 | 67090 | 84.2% |
| transit small | 2..10 | 9977 | 12.5% |
| transit middle | 11..1000 | 2511 | 3.2% |
| transit large | 1001..10000 | 55 | 0.1% |
| transit huge | 10001..-1 | 11 | 0.0% |

279

### Question 1

What percentage of ASNs are edge ASes (customer cone size of 1)? What does this suggest about the structure of the Internet?

84.2%. This suggests that the vast majority of ASs are small personal ASs. The internet is structured like a snowflake, with the center of the snowflake being composed of the largest providers, and the edges of the snowflake being supported by the inner structure while composing the majority of the snowflake. 

### Question 2

How large is the maximum customer cone? What does this tell you about the most influential ASes on the Internet?

56415 is the size of the largest cone. The most influenctial ASes serve a large majority of the internet, with this largest one having almost three quarters of the internet in it's customer cone.

### Question 3

How do the proportions of edge, small transit, and large transit ASes compare? What does this distribution reveal about how ASes are organized hierarchically?

The largest ASes are the fewest, and the smallest are the most numerous. The big ASes are few, but serve a very large number of smaller edge and small transit ASes. The proportion of small transit to edge ASes suggests that each small transit AS serves 6 edge ASes on average, while large transit ASes (assuming they don't serve edge ASes directly) serve almost 200 small transit ASes. 

---

## Task 3: How Are ASN Tiers Divided Across Countries?

Fill in the missing code in the cells below to produce **Table 2**.

- Use the tier classification from Task 2 (`classify()` and `TIERS`).
- Use `data/as2org.jsonl` to map each ASN to its organization's 2-letter country code.
  Each line is a JSON object with a `"country"` field and a `"members"` list of ASN strings.
- Find the top 4 countries by total ASN count across all tiers.
  All other countries map to **other**.
- The top-4 country codes become the dynamic column headers (e.g. `US`, `CN`).

### Table 2 format

| name           | 1st           | 2nd           | 3rd           | 4th           | other         |
| -------------- | ------------- | ------------- | ------------- | ------------- | ------------- |
| edge           | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit small  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit middle | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit large  | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |
| transit huge   | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) | [total] ([%]) |

Replace `1st`/`2nd`/`3rd`/`4th` with the actual 2-letter country codes.
Each cell shows `total (X.X%)`, where `X.X%` is that country's share of the tier's total.

In [56]:
AS2ORG_URL = "http://rook-ceph-rgw-nautiluss3.rook/caida/as2org/as2org.jsonl"
AS2ORG_PATH = Path("data/as2org.jsonl")

print(f"Downloading {AS2ORG_URL} ...")
AS2ORG_PATH.parent.mkdir(parents=True, exist_ok=True)  # ensure 'data/' exists
urllib.request.urlretrieve(AS2ORG_URL, AS2ORG_PATH)
if not AS2ORG_PATH.exists(): 
    print (f"Unable to save file {AS2ORG_PATH}")
    exit() 
    
asn_to_country = {}
with open_safe(AS2ORG_PATH) as fin:
    print(fin.readline())
    for line in fin:
        line = line.strip()
        if not line:
            continue
        jline = json.loads(line)
        country = jline.get('country')
        members = jline.get('members')
        for asn in members:
            asn_to_country[asn] = country
        # Parse the JSON line, extract the ASN's in the members list
        # and country code, and add it to the asn_to_country dict.
print (f"number of ASNs with countries: {len(asn_to_country)}")

{"score":120793,"orgId":"589f9199b0","orgName":"Level 3 Parent, LLC","country":"US","source":"ARIN","members":["1","189","199","200","201","202","203","279","281","560","594","595","596","597","598","2551","3356","3508","3549","3831","4323","6395","6467","6484","7037","7176","7911","8043","10753","11213","16852","18756","19094","19591","19962","20476","22026","26458","30686","32421"],"changed":"2024-06-17T00:00:00+00:00","date":"2026-04-01T00:00:00+00:00","ts":"2026-04-17T11:48:09+00:00"}

number of ASNs with countries: 120753
1299 SE
1833 SE
8233 SE
8532 SE
8837 SE
286 US
517 US
2858 US
3257 US
3291 US


In [84]:
from collections import defaultdict

# Count ASNs per (tier, country) pair
tier_country_counts = defaultdict(lambda: defaultdict(int))
country_totals = defaultdict(int)

for asn, size in asn_to_size.items():
    tier = classify(size)
    country = asn_to_country.get(asn, "unknown")
    # YOUR CODE HERE: increment tier_country_counts[tier][country] and country_totals[country]
    tier_country_counts[tier][country] += 1
    country_totals[country]+=1

# Find top-4 countries by total ASN count
top4 = [c for c, _ in sorted(country_totals.items(), key=lambda x: -x[1])[:4]]
columns = top4 + ["other"]

# Build Markdown table
header = "| tier | " + " | ".join(columns) + " |"
sep    = "| --- | " + " | ".join(["---"] * len(columns)) + " |"
table_2_rows = [header, sep]

# YOUR CODE HERE: for each (label, _, _) in TIERS, append a formatted row to table_2_rows
#   each cell: f"{n} ({pct:.1f}%)" where pct is the country's share of that tier's total
#   sum counts for countries not in top4 into "other"
for tier, _, _ in TIERS:
    row = f'|{tier}|'
    sum_pct = 100
    for country in top4:
        pct = tier_country_counts[tier][country] / tier_counts[tier] * 100
        sum_pct -= pct
        row += f"{pct:.1f}|"
    row += f"{sum_pct:.1f}|"
    table_2_rows.append(row)

table_2 = "\n".join(table_2_rows)

In [83]:
display(Markdown(table_2))

| tier | US | BR | RU | IN | other |
| --- | --- | --- | --- | --- | --- |
|edge|24.7|8.9|6.2|3.7|56.6|
|transit small|15.8|19.1|6.9|3.2|55.1|
|transit middle|16.4|13.9|5.1|2.6|62.0|
|transit large|18.2|12.7|14.5|5.5|49.1|
|transit huge|54.5|0.0|0.0|0.0|45.5|

### Question 4

Which countries have the most transit huge ASNs? What does this tell you about where Internet infrastructure is concentrated?

The US has over half of the huge ASNs. Clearly, internet infrastructure is concentrated in the US

### Question 5

What proportion of all ASNs fall in the "other" category? What does this suggest about the geographic distribution of the global Internet?

Roughly half of ASNs fall into the other category. This means that most internet infrastructure is concentrated in 4 countries, and the rest of the infrastructure is spread out in the rest of the world.

### Question 6

Do the same countries dominate across all AS tiers (edge, transit small, transit huge)? What patterns do you observe across the rows?

Mostly the US has the most ASes across tiers, although Brazil has the most small transit ASes. Generally, the US and Brazil dominate the categories, and Russia and India have significantly less than US and Brazil. The large transit category is mnore evenly divided between US, Brazil, and Russia, with India still having a significantly lower amount. Brazil and Russia have higher proportions of larger ASes, which suggests that they have a lot of larger infrastructure, but less consumers.